# Mixture of Experts

As opposed to a dense transformer infrastructure where every parameter is activated, a Mixture of expert infrastructure's FFN mechanism is divided into many "experts" that are activated sparsely.

<div align="center">
  <img src="../Medias/MoE.png" width="1200">
</div>

To be more specific, instead of having a single FFN for each FFN layer, there consist of several "expert" FFN where each word is routed to. In essense, you have more parameters and expressiveness without effecting computes. When there are many experts as well, each experts can live on separate device and utulize the efficiency of paralellism.

The most common way of routing is called "token choose experts," where each token and their entire corresponding embedding vector is routed to expert(s). The amount of experts that they are routed to depends on the hyperparameter $K$, and each token chooses their top-$K$ number of experts based on the following procedure:

$$E = [e_1, e_2, ..., e_N]^T$$
$$E \in \mathbb{R}^{N \times d_{model} }$$

Where $E$ consist of $n$ numbers of experts embedding vectors. The routing is decided via the following process:

$$S = \text{Softmax}_{Column}(EX^T)$$
$$S \in \mathbb{R}^{N \times seq}$$

Intuatively, we are forming a new matrix where each token, $x_t$, is compared to each expert $e_i$ via the dot product. We then softmax it column wise such that for each token, so that we get a probability for each experts for a given token. Then, for each token, we pick the top-K:

$$G_{i,t} = \begin{cases} 
S_{i,t}, & S_{i,t} \in \text{Topk}(S, K), \\ 
0, & \text{otherwise}, 
\end{cases}$$

The final expression is then a weighed average of all the chosen experts, given as:

$$y_{t} = \sum_{i=1}^N \Big(G_{i,t} \cdot FFN_i (x_t) \Big) + x_t$$

Note that the addition of $x_i$ is due to the residual stream. $G$ is nondifferentiable, which pose a problem during backpropagation. During backpropagration, we assume $G = S$.

<div align="center">
  <img src="../Medias/MoErouting.png" width="600">
</div>

Originally, MoE was based off of the dense model, where each experts was a copy of the dense model with all its dimenionality the same. Note that there are many variations within the design of this architecture however. For example many models like Deepseek chooses to have 7 routed experts and also a shared expert. 

There also exist fine-grained expert architecture, where we make each experts smaller (Whereas we typically multiplied our hidden layer by 4, we multiply it by 2 instead) instead of duplicating the number of parameters we have, such that we get more experts while not incurring the cost of additional parameters. If we started with 8 experts for example, we can make each experts twice as small while doubling our K value. Even though we are picking more experts, the total number of parameters activated remains the same because each individual expert is much smaller.

In order to train a MoE model, we want to modify each $e_i$ value to mimimize our loss. The auxiliary loss is given by:

$$\mathcal{L} = \alpha \cdot N \sum_{i=1}^N f_i P_i$$
$$f_i = \frac{1}{T} \sum_{x \in \mathcal{B}} 1\{\argmax p(x) = i\}$$
$$P_i = \frac{1}{T} \sum_{x \in \mathcal{B}} p_i(x)$$

Where $\mathcal{B}$ is the batch of input sequences, and $T$ is the total number of tokens within the batch. The $f$ vector counts the total proportion that an expert was chosen across the batch, while $P$ calculates the fraction of router probability that was allocated to a particular expert. Together, the Loss term sought spread out the experts that are choosen, and prevents the router from choosing the same experts over and over again (and thus prevent the model from getting stuck in a local minimum).

Keep in mind too that if an expert is not selected by the router for a specific input, its weights are not altered for that step, thus spreading out which experts get used help to train every expert instead of only some. We see this with the following:

$$\frac{\partial \mathcal{L}}{\partial p_i(x_t)} = \frac{\alpha N}{T} f_i$$

Where we are taking the partial of the loss with respect to a particular $p_i(x_t)$ value corresponding to some token $t$. So if a particular expert is getting too many tokens, then during gradient descent it will be more negatively pushed downward.

Another way to train an MoE is to Upcycle it, where a regular dense model is trained, then split/duplicate the FFN layer into many experts and introducing a router.